# Explore here

It's recommended to use this notebook for exploration purposes.

In [26]:
import os
from bs4 import BeautifulSoup
import requests
import time
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd



In [49]:
import re

url = 'https://en.wikipedia.org/wiki/List_of_Spotify_streaming_records'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')

ths = soup.find_all('th')
tds = soup.find_all('td')

songs_title = []
artists_name = []
streams = []
realease_date = []


# Get from ths the songs title
for th in ths:
    if th.find('a'):
        songs_title.append(th.find('a').text)

# Get from tds the artists name and the number of streams
for td in tds:
    if td.find('a') and len(td.find('a').text) <= 3:
        continue
    elif td.find('a'):
        artists_name.append(td.find('a').text)
    elif td.find('span'):
        realease_date.append(td.find('span').text)
    else:
        text = td.text.strip().replace('.', '').replace(',', '')
        if text.isdigit() and int(text) > 101:  # We want to avoid the number of streams that are in the ths
            streams.append(int(text))

#Clean to get only de 100 songs
songs_title = songs_title[:100]
artists_name = artists_name[:100]
streams = streams[:100]
realease_date = realease_date[:100]


df = pd.DataFrame({
    'song_title': songs_title,
    'artist_name': artists_name,
    'streams': streams,
    'release_date': realease_date
})

df.head()


,song_title,artist_name,streams,release_date
0,Blinding Lights,The Weeknd,5386,29 November 2019
1,Shape of You,Ed Sheeran,4882,6 January 2017
2,Sweater Weather,The Neighbourhood,4560,3 December 2012
3,Starboy,The Weeknd,4501,21 September 2016
4,As It Was,Harry Styles,4380,1 April 2022


In [50]:
conn = sqlite3.connect('spotify_streaming_records.db')
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS spotify_records (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    song_title TEXT,
                    artist_name TEXT,
                    streams INTEGER,
                    release_date TEXT
                )''')
for index, row in df.iterrows():
    cursor.execute('''INSERT INTO spotify_records (song_title, artist_name, streams, release_date)
                      VALUES (?, ?, ?, ?)''', (row['song_title'], row['artist_name'], row['streams'], row['release_date']))
    
#I found that pandas has a better way to insert data in sql:
# df.to_sql('spotify_records', conn, if_exists='replace', index=False)

conn.commit()
conn.close()